# **Cross-Fitting S-learner**

The S-learner (Single learner) is the most straightforward way an ML model estimates casual effects (marketing campaign or discount actually causes a customer to stay)

## **The Churn Scenario Setup**

Imagine you run a subscription software company and want to know: Does offering a $20 discount code ($X$) actually prevent customers from churning ($Y$)?

In your historical data, you have:
- Treatment ($X$): $1$ if the customer received a discount, $0$ if they did not.
- Outcome ($Y$): $1$ if the customer churned (canceled), $0$ if they stayed. Because $Y$ is binary, our machine learning model will be a classifier (like XGBoost or Logistic Regression) that predicts the probability of churning ($P(Y=1)$).
- Confounders ($Z_1, Z_2$): Variables that affect both whether a customer gets a discount and whether they churn.
    - $Z_1$: Customer tenure (months subscribed).
    - $Z_2$: App logins per week.
    
(Why are $Z_1$ and $Z_2$ confounders? Because your retention team historically gave discounts mostly to low-usage, high-risk customers, creating selection bias.)

## **Implementation**

### **Training and Testing of the Model**

In [91]:
# Import necessary libraries
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split,GridSearchCV, StratifiedKFold
from sklearn.metrics import  recall_score, precision_score, f1_score,roc_auc_score

**Generate synthetic data for demonstration**

In [92]:
def generate_enterprise_churn_data(n=10000, seed=42):
    np.random.seed(seed)
    
    # 1. Realistic Confounders
    # Tenure: Gamma distribution (skewed: most customers are newer, fewer are multi-year veterans)
    tenure_months = (np.clip(np.random.gamma(shape=2, scale=6, size=n), 1, 60)).astype(int)
    
    # Weekly Logins: Poisson distribution
    weekly_logins = np.random.poisson(lam=4, size=n)
    
    # Support Tickets: Higher tickets usually indicate friction
    support_tickets = np.random.poisson(lam=1.5, size=n)
    
    # Monthly Spend Tier ($10, $25, $50, $100)
    monthly_spend = np.random.choice([10, 25, 50, 100], size=n, p=[0.4, 0.3, 0.2, 0.1])
    
    # 2. Confounding Mechanism (Propensity Score)
    # Retention teams historically give discounts to customers with low tenure, low logins, AND high support tickets.
    propensity_logit = -0.5 + (0.05 * tenure_months) - (0.20 * weekly_logins) + (0.40 * support_tickets) - (0.005 * monthly_spend)
    discount_prob = 1 / (1 + np.exp(-propensity_logit))
    treatment_x = np.random.binomial(1, discount_prob)
    
    # 3. Outcome Mechanism (Churn)
    # True Causal Effect: Offering a discount reduces the log-odds of churning by -0.35 (~15-20% absolute drop)
    true_causal_effect = -0.35
    
    baseline_logit = (
        1.2 
        - (0.06 * tenure_months) 
        - (0.15 * weekly_logins) 
        + (0.50 * support_tickets) 
        - (0.01 * monthly_spend)
        + (true_causal_effect * treatment_x) # The true causal impact!
    )
    churn_prob = 1 / (1 + np.exp(-baseline_logit))
    outcome_y = np.random.binomial(1, churn_prob)
    
    df = pd.DataFrame({
        'tenure_months': tenure_months,
        'weekly_logins': weekly_logins,
        'support_tickets': support_tickets,
        'monthly_spend': monthly_spend,
        'discount_given': treatment_x, # Treatment (X)
        'churned': outcome_y           # Outcome (Y)
    })
    
    return df

In [93]:
df = generate_enterprise_churn_data(n=10000)

In [94]:
df.head()

,tenure_months,weekly_logins,support_tickets,monthly_spend,discount_given,churned
0,14,7,1,25,0,0
1,8,4,3,50,0,0
2,8,9,2,10,1,1
3,8,7,0,10,0,0
4,27,3,1,100,0,1


**Split the data**

In [95]:
# train test split
X = df.drop(columns=['churned'])
y = df['churned']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=50)

In [96]:
print(f"Dataset Split Sizes:")
print(f" - Train Set: {len(X_train)} rows")
print(f" - Test Set: {len(X_test)} rows\n")

Dataset Split Sizes:
 - Train Set: 8000 rows
 - Test Set: 2000 rows



**Train Cross-fitting S-learner**

In [97]:
# from sklearn import metrics
# metrics.get_scorer_names()

In [ ]:
# Define Model
# https://scikit-learn.org/1.8/modules/generated/sklearn.ensemble.RandomForestClassifier.html
rf = RandomForestClassifier(max_depth=10, n_estimators=250, random_state=50)
# You can use different values for n_estimators, you would see that the ATE varies for each value
# rf = RandomForestClassifier(max_depth=10, n_estimators=120, random_state=50) 

# Define Cross-Validation Strategy
# This takes care of train and validaition splits for us
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=50)

all_individual_effects = []
oof_predictions_proba = np.zeros(len(X_train))
oof_actuals = y_train.values

print("Starting Cross-Fitting across 5 folds...")

for fold, (train_idx, val_idx) in enumerate(cv.split(X_train, y_train)):
    # Split fold data
    X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
    X_val, y_val = X_train.iloc[val_idx], y_train.iloc[val_idx]

    # Train the model on the training fold
    fold_model  = rf
    fold_model.fit(X_tr, y_tr)

    # Store out-of-fold predictions
    oof_predictions_proba[val_idx] = fold_model.predict_proba(X_val)[:, 1]

    # Counterfactual Simulation on the validation fold
    X_val_1 = X_val.copy()
    X_val_1['discount_given'] = 1  # Simulate treatment
    prob_y1 = fold_model.predict_proba(X_val_1)[:, 1]

    X_val_0 = X_val.copy()
    X_val_0['discount_given'] = 0  # Simulate control
    prob_y0 = fold_model.predict_proba(X_val_0)[:, 1]

    # Calculate fold causal effects
    fold_effects = prob_y1 - prob_y0
    all_individual_effects.extend(fold_effects)
    # all_individual_effects.append(fold_effects)


print("Cross-Fitting Complete!\n")

# Calculate Cross-Fitted ATE
cross_fitted_ate = np.round(np.mean(all_individual_effects),2)
print(f"Cross-Fitted Training ATE (Validation set): {cross_fitted_ate}")

# Evaluate Out-Of-Fold Predictive Metrics
oof_preds_binary = (oof_predictions_proba >= 0.5).astype(int)
print("Out-Of-Fold Metrics (Validation Set):")
print(f" - ROC-AUC:   {np.round(roc_auc_score(oof_actuals, oof_predictions_proba),2)}")
print(f" - Recall:    {np.round(recall_score(oof_actuals, oof_preds_binary),2)}")
print(f" - Precision: {np.round(precision_score(oof_actuals, oof_preds_binary),2)}")
print(f" - F1-Score:  {np.round(f1_score(oof_actuals, oof_preds_binary),2)}\n")

Starting Cross-Fitting across 5 folds...
Cross-Fitting Complete!

Cross-Fitted Training ATE (Validation set): -0.06
Out-Of-Fold Metrics (Validation Set):
 - ROC-AUC:   0.7
 - Recall:    0.71
 - Precision: 0.66
 - F1-Score:  0.68



In [99]:
print("Training final model on the entire training set...")
final_model = rf
final_model.fit(X_train, y_train)

Training final model on the entire training set...


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",250
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y

**Evaluate on test data**

In [100]:
# The 1 means we are taking the probability of the positive class (churn = 1)
# Evaluate predictive performance on the untouched Test Set
test_prediction_proba = final_model.predict_proba(X_test)[:, 1]
test_prediction = final_model.predict(X_test)

In [101]:
# Evaluations
test_auc = roc_auc_score(y_test, test_prediction_proba)
test_recall = recall_score(y_test, test_prediction)
test_precision = precision_score(y_test, test_prediction)
test_f1 = f1_score(y_test, test_prediction)

In [102]:
print("Test Set Evaluation Metrics:")
print(f"ROC-AUC: {np.round(test_auc, 2)}")
print(f"Recall: {np.round(test_recall, 2)}")
print(f"Precision: {np.round(test_precision, 2)}")
print(f"F1-Score: {np.round(test_f1, 2)}")

Test Set Evaluation Metrics:
ROC-AUC: 0.71
Recall: 0.72
Precision: 0.67
F1-Score: 0.69


### **Counterfactuals (ATE): S-Learner Approach**

So the model has been trained and tested. Now we Simulate A/B groups 

In [103]:
# Group A: Discount Given
test_x1 = X_test.copy()
test_x1['discount_given'] = 1
prob_y1 = final_model.predict_proba(test_x1)[:, 1]

In [104]:
# Group B: No Discount Given
test_x2 = X_test.copy()
test_x2['discount_given'] = 0
prob_y2 = final_model.predict_proba(test_x2)[:, 1]

In [105]:
# Calculate Average Treatment Effect (ATE)
ate = np.round(np.mean(prob_y1 - prob_y2),3)
print(f"Average Treatment Effect (ATE) of Discount on Churn Probability: \n{ate}")

Average Treatment Effect (ATE) of Discount on Churn Probability: 
-0.058


**Inspection of Prediction group A**

In [106]:
test_x1['predicted_churn'] = final_model.predict(test_x1)

In [107]:
test_x1['actual_discount'] = X_test['discount_given']
test_x1['actual_churn'] = y_test
test_x1.head()

,tenure_months,weekly_logins,support_tickets,monthly_spend,discount_given,predicted_churn,actual_discount,actual_churn
9102,35,6,1,25,1,0,0,0
7868,5,2,2,50,1,1,0,1
4176,5,2,0,100,1,0,0,0
4161,4,2,1,10,1,1,1,1
8770,6,4,1,25,1,0,0,1


**Inspection of Prediction group B**

In [108]:
test_x2['predicted_churn'] = final_model.predict(test_x2)

In [109]:
test_x2['actual_discount'] = X_test['discount_given']
test_x2['actual_churn'] = y_test
test_x2.head()

,tenure_months,weekly_logins,support_tickets,monthly_spend,discount_given,predicted_churn,actual_discount,actual_churn
9102,35,6,1,25,0,0,0,0
7868,5,2,2,50,0,1,0,1
4176,5,2,0,100,0,0,0,0
4161,4,2,1,10,0,1,1,1
8770,6,4,1,25,0,1,0,1


### **Interpretation of ATE**

Since estimated Average Treatment Effect (ATE) is **-0.058** (or roughly **-5.8%**), it means:

> **Offering the discount causes an absolute 5.8% reduction in the probability of customer churn**, on average, after properly controlling for all confounders (like tenure, app logins, support tickets, and monthly spend).

So what does it actually mean?

1. **Direction (Negative):** Because churn represents "leaving" ($1 = \text{churned}, 0 = \text{stayed}$), a **negative** causal effect is a **good thing**. It means the discount is working; it pushes the probability of canceling downward.
2. **Magnitude (5.8 Percentage Points):** It is an *absolute percentage point change*, not a relative percentage. For example:
    - If a customer's baseline predicted churn risk without a discount was **20.0%**, giving them the discount drops their predicted churn risk to **14.2%** ($20.0\% - 5.8\%$).

3. **Causally Isolated:** Because the S-learner (backed by the machine learning model and cross-validation) controlled for selection bias, this isn't just a coincidence or correlation. It proves that the retention team's discount program actively saved customers who might have otherwise left.